# 4. Backend (FastAPI) — Colab version

Runs the FastAPI server *inside* this Colab notebook and exposes it publicly via ngrok, so `frontend/index.html` running on your own machine can reach it.

Requires `models/best_sign_language_model.pt` and `models/vocab.pkl` from notebook 3 to be present in this Colab runtime (either trained in this same session, or copied back from Google Drive).

In [ ]:
!pip install -q fastapi uvicorn python-multipart nest_asyncio pyngrok mediapipe==0.10.21 opencv-python

## Restart runtime if you just changed the mediapipe version
Then re-run from here.

In [ ]:
import os
import pickle
import shutil
import tempfile
import threading

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import mediapipe as mp
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Model + feature-extraction code (same as notebooks 2 & 3)

In [ ]:
mp_holistic = mp.solutions.holistic

def _landmarks_to_array(landmarks, n_points, n_dims):
    if landmarks is None:
        return np.zeros(n_points * n_dims, dtype=np.float32)
    if n_dims == 4:
        vals = [[p.x, p.y, p.z, p.visibility] for p in landmarks.landmark]
    else:
        vals = [[p.x, p.y, p.z] for p in landmarks.landmark]
    return np.array(vals, dtype=np.float32).flatten()

def extract_keypoints(results):
    pose = _landmarks_to_array(results.pose_landmarks, 33, 4)
    face = _landmarks_to_array(results.face_landmarks, 468, 3)
    lh = _landmarks_to_array(results.left_hand_landmarks, 21, 3)
    rh = _landmarks_to_array(results.right_hand_landmarks, 21, 3)
    return np.concatenate([pose, face, lh, rh])

def extract_features_from_video(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    frame_features = []
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = holistic.process(image)
            frame_features.append(extract_keypoints(results))
            if max_frames and len(frame_features) >= max_frames:
                break
    cap.release()
    if len(frame_features) == 0:
        return np.zeros((1, 1662), dtype=np.float32)
    return np.stack(frame_features).astype(np.float32)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=1662, hid_dim=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.input_proj = nn.Linear(input_dim, hid_dim)
        self.lstm = nn.LSTM(hid_dim, hid_dim, n_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc_h = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, x, lengths):
        x = self.dropout(self.input_proj(x))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, (hidden, cell) = self.lstm(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        hidden = hidden.view(self.n_layers, 2, hidden.size(1), hidden.size(2))
        hidden = torch.cat((hidden[:, 0], hidden[:, 1]), dim=2)
        hidden = self.fc_h(hidden)
        cell = cell.view(self.n_layers, 2, cell.size(1), cell.size(2))
        cell = torch.cat((cell[:, 0], cell[:, 1]), dim=2)
        cell = self.fc_c(cell)
        return outputs, hidden, cell


class Attention(nn.Module):
    def __init__(self, enc_hid_dim=512, dec_hid_dim=512):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size, src_len, _ = encoder_outputs.shape
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        weights = torch.softmax(attention, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim=256, enc_hid_dim=512, dec_hid_dim=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.attention = Attention(enc_hid_dim, dec_hid_dim)
        self.lstm = nn.LSTM(emb_dim + (enc_hid_dim * 2), dec_hid_dim, n_layers, batch_first=True, dropout=dropout)
        self.fc_out = nn.Linear(dec_hid_dim + (enc_hid_dim * 2) + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        context, attn_w = self.attention(hidden[-1], encoder_outputs)
        context = context.unsqueeze(1)
        lstm_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        context = context.squeeze(1)
        prediction = self.fc_out(torch.cat((output, context, embedded), dim=1))
        return prediction, hidden, cell, attn_w


class SignLanguageTranslator(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device


def translate(model, features, vocab, device, max_len=50):
    model.eval()
    if isinstance(features, np.ndarray):
        features = torch.FloatTensor(features)
    features = features.unsqueeze(0).to(device)
    src_len = torch.LongTensor([features.shape[1]]).to(device)
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(features, src_len)
    inputs = torch.LongTensor([vocab.word2idx["<sos>"]]).to(device)
    outputs = []
    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell, _ = model.decoder(inputs, hidden, cell, encoder_outputs)
        pred = output.argmax(1).item()
        if pred == vocab.word2idx["<eos>"]:
            break
        outputs.append(vocab.idx2word[pred])
        inputs = torch.LongTensor([pred]).to(device)
    return " ".join(outputs)

## Load the trained model + vocab

In [ ]:
MODEL_DIR = "models"
CHECKPOINT_PATH = os.path.join(MODEL_DIR, "best_sign_language_model.pt")
VOCAB_PATH = os.path.join(MODEL_DIR, "vocab.pkl")
HID_DIM, ENC_LAYERS, DEC_LAYERS = 512, 2, 2

if not os.path.exists(VOCAB_PATH) or not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Couldn't find {CHECKPOINT_PATH} / {VOCAB_PATH}. "
        "Run notebook 03_train first, or copy the model files from Google Drive into this runtime."
    )

with open(VOCAB_PATH, "rb") as f:
    vocab = pickle.load(f)

enc = Encoder(input_dim=1662, hid_dim=HID_DIM, n_layers=ENC_LAYERS, dropout=0.0)
dec = Decoder(output_dim=len(vocab), emb_dim=256, enc_hid_dim=HID_DIM, dec_hid_dim=HID_DIM, n_layers=DEC_LAYERS, dropout=0.0)
model = SignLanguageTranslator(enc, dec, device).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()
print(f"Model loaded on {device}, vocab size {len(vocab)}")

## Define the FastAPI app

In [ ]:
app = FastAPI(title="Sign Language Translator API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    suffix = os.path.splitext(file.filename or "")[1] or ".mp4"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        shutil.copyfileobj(file.file, tmp)
        tmp_path = tmp.name
    try:
        features = extract_features_from_video(tmp_path)
        prediction = translate(model, features, vocab, device)
    finally:
        os.remove(tmp_path)
    return {"prediction": prediction}

## Run the server in the background

In [ ]:
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("Server starting on port 8000...")

## Expose it publicly with ngrok
You'll need a free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken — paste it below.

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- paste your token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)
print("Public backend URL:", public_url)
print("Paste this into the 'backend' field at the bottom of frontend/index.html")

## Quick test from within the notebook
Upload a short video to test without needing the frontend at all.

In [ ]:
from google.colab import files
import requests

uploaded = files.upload()
video_filename = next(iter(uploaded))

with open(video_filename, "rb") as f:
    resp = requests.post("http://localhost:8000/predict", files={"file": f})

print(resp.status_code, resp.json())